In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import math
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ========== CONFIG ==========
MODEL_PATH = r"C:\Users\gayat\Downloads\ACV_projects-20260306T053026Z-1-001\ACV_GESTURE PROJECT\hand_landmarker.task"
DATASET_PATH = "air_dataset"
PINCH_THRESHOLD = 0.04
RESAMPLE_POINTS = 100

# ========== CREATE DATASET FOLDER ==========
if not os.path.exists(DATASET_PATH):
    os.makedirs(DATASET_PATH)

# ========== LOAD MODEL ==========
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    running_mode=vision.RunningMode.IMAGE
)

detector = vision.HandLandmarker.create_from_options(options)

# ========== PREPROCESS FUNCTION (INTERPOLATION) ==========
def preprocess_stroke(points, num_points=100):
    if len(points) < 20:
        return None

    points = np.array(points)

    # Remove translation (center)
    points = points - np.mean(points, axis=0)

    # Scale normalize
    max_val = np.max(np.abs(points))
    points = points / (max_val + 1e-6)

    # Interpolation resampling
    t_original = np.linspace(0, 1, len(points))
    t_new = np.linspace(0, 1, num_points)

    x_interp = np.interp(t_new, t_original, points[:, 0])
    y_interp = np.interp(t_new, t_original, points[:, 1])

    resampled = np.stack((x_interp, y_interp), axis=1)

    return resampled.flatten()

# ========== CAMERA ==========
cap = cv2.VideoCapture(0)

drawing = False
points = []

print("Instructions:")
print("Pinch (thumb + index) to draw")
print("Press S to save letter")
print("Press C to clear")
print("Press Q to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    result = detector.detect(mp_image)

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            h, w, _ = frame.shape

            thumb = hand_landmarks[4]
            index = hand_landmarks[8]

            thumb_x, thumb_y = int(thumb.x * w), int(thumb.y * h)
            index_x, index_y = int(index.x * w), int(index.y * h)

            # Draw landmarks
            for lm in hand_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)

            # Pinch detection
            distance = math.sqrt((thumb.x - index.x)**2 + (thumb.y - index.y)**2)

            if distance < PINCH_THRESHOLD:
                drawing = True
                points.append((index_x, index_y))
            else:
                drawing = False

    # Draw trajectory
    for i in range(1, len(points)):
        cv2.line(frame, points[i - 1], points[i], (255, 0, 0), 4)

    cv2.imshow("Air Writing Dataset Builder", frame)

    key = cv2.waitKey(1) & 0xFF

    # SAVE SAMPLE
    if key == ord('s'):
        features = preprocess_stroke(points, RESAMPLE_POINTS)

        if features is not None:
            label = input("Enter letter label: ").upper()

            label_folder = os.path.join(DATASET_PATH, label)

            if not os.path.exists(label_folder):
                os.makedirs(label_folder)

            sample_count = len(os.listdir(label_folder))
            file_path = os.path.join(label_folder, f"sample_{sample_count+1}.npy")

            np.save(file_path, features)

            print(f"Saved {file_path}")

        else:
            print("Not enough points, try again.")

        points = []

    # CLEAR CURRENT DRAWING
    if key == ord('c'):
        points = []

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Instructions:
Pinch (thumb + index) to draw
Press S to save letter
Press C to clear
Press Q to quit
Saved air_dataset\sample_1.npy
Saved air_dataset\L\sample_1.npy


In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ========== CONFIG ==========
DATASET_PATH = r"C:\Users\gayat\Downloads\ACV_projects-20260306T053026Z-1-001\ACV_GESTURE PROJECT\air_dataset"

# ========== LOAD DATA ==========
X = []
y = []

for label in os.listdir(DATASET_PATH):
    label_folder = os.path.join(DATASET_PATH, label)
    if os.path.isdir(label_folder):
        for file in os.listdir(label_folder):
            if file.endswith(".npy"):
                data = np.load(os.path.join(label_folder, file))
                X.append(data)
                y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

# ========== TRAIN / TEST SPLIT ==========
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ========== AUGMENTATION FUNCTION ==========
def augment_sample(sample):
    sample = sample.reshape(-1, 2)

    # Small rotation
    angle = np.random.uniform(-10, 10) * np.pi / 180
    rotation_matrix = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)]
    ])
    sample = sample @ rotation_matrix.T

    # Small Gaussian noise
    sample += np.random.normal(0, 0.01, sample.shape)

    return sample.flatten()

# ========== APPLY AUGMENTATION TO TRAIN SET ==========
augmented_X = []
augmented_y = []

for i in range(len(X_train)):
    augmented_X.append(X_train[i])
    augmented_y.append(y_train[i])

    # Add 1 augmented version
    augmented_X.append(augment_sample(X_train[i]))
    augmented_y.append(y_train[i])

X_train = np.array(augmented_X)
y_train = np.array(augmented_y)

print("After augmentation train size:", X_train.shape)

# ========== FEATURE SCALING ==========
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ========== TRAIN SVM ==========
model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
model.fit(X_train, y_train)

# ========== EVALUATE ==========
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ========== SAVE MODEL ==========
joblib.dump(model, "svm_air_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel saved successfully.")

Dataset shape: (202, 200)
Labels: ['A' 'C' 'L' 'V']
After augmentation train size: (322, 200)

Accuracy: 1.0

Confusion Matrix:
 [[10  0  0  0]
 [ 0 10  0  0]
 [ 0  0 11  0]
 [ 0  0  0 10]]

Classification Report:

              precision    recall  f1-score   support

           A       1.00      1.00      1.00        10
           C       1.00      1.00      1.00        10
           L       1.00      1.00      1.00        11
           V       1.00      1.00      1.00        10

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41


Model saved successfully.


In [2]:
import os
import json
import numpy as np
import time
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ================= CONFIG =================
BASE_PATH = r"C:\Users\gayat\Downloads\archive (8)"
TRAIN_PATH = os.path.join(BASE_PATH, "ann_train_val")
TEST_PATH = os.path.join(BASE_PATH, "ann_test")

# Only keep these gestures
ALLOWED_CLASSES = {"like", "palm", "fist", "peace", "call", "stop"}

# ================= DATA LOADER =================
def load_dataset(folder_path):
    X = []
    y = []

    for file in os.listdir(folder_path):
        if not file.endswith(".json"):
            continue

        class_name = file.replace(".json", "")

        # Filter unwanted classes
        if class_name not in ALLOWED_CLASSES:
            continue

        file_path = os.path.join(folder_path, file)

        with open(file_path, "r") as f:
            data = json.load(f)

        for sample_id, sample in data.items():

            if "landmarks" not in sample:
                continue

            landmarks_all = sample["landmarks"]

            if not landmarks_all:
                continue

            landmarks = landmarks_all[0]

            if len(landmarks) != 21:
                continue

            landmarks = np.array(landmarks, dtype=np.float32)

            if landmarks.shape != (21, 2):
                continue

            leading_hand = sample.get("leading_hand", "right")

            # Hand-invariant
            if leading_hand == "left":
                landmarks[:, 0] = 1.0 - landmarks[:, 0]

            features = landmarks.flatten()

            X.append(features)
            y.append(class_name)

    return np.array(X), np.array(y)

# ================= LOAD DATA =================
print("Loading filtered training data...")
X_train, y_train = load_dataset(TRAIN_PATH)

print("Loading filtered test data...")
X_test, y_test = load_dataset(TEST_PATH)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", np.unique(y_train))

# ================= SCALE =================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ================= TRAIN =================
print("\nTraining LinearSVC...")
start_time = time.time()

model = LinearSVC(C=1.0, max_iter=5000)
model.fit(X_train, y_train)

print("Training time:", round(time.time() - start_time, 2), "seconds")

# ================= EVALUATE =================
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ================= SAVE =================
joblib.dump(model, "gesture_control_svm.pkl")
joblib.dump(scaler, "gesture_control_scaler.pkl")

print("\nFiltered gesture control model saved successfully.")

Loading filtered training data...
Loading filtered test data...
Train shape: (164595, 42)
Test shape: (14211, 42)
Classes: ['call' 'fist' 'like' 'palm' 'peace' 'stop']

Training LinearSVC...
Training time: 20.56 seconds

Accuracy: 0.8981774681584688

Confusion Matrix:
 [[2197   17   67   42   10   52]
 [  27 2133   56   64   27  106]
 [  26   20 2071   75   18  111]
 [  16   19   62 2143   22  113]
 [  11   22   61   99 2093   97]
 [  19   21   51   97   19 2127]]

Classification Report:
               precision    recall  f1-score   support

        call       0.96      0.92      0.94      2385
        fist       0.96      0.88      0.92      2413
        like       0.87      0.89      0.88      2321
        palm       0.85      0.90      0.88      2375
       peace       0.96      0.88      0.92      2383
        stop       0.82      0.91      0.86      2334

    accuracy                           0.90     14211
   macro avg       0.90      0.90      0.90     14211
weighted avg      

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
import joblib
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ================= LOAD MODELS =================
air_model = joblib.load("svm_air_model.pkl")
air_scaler = joblib.load("scaler.pkl")

gesture_model = joblib.load("gesture_control_svm.pkl")
gesture_scaler = joblib.load("gesture_control_scaler.pkl")

GESTURE_CLASSES = gesture_model.classes_

# ================= CONFIG =================
HAND_MODEL_PATH = r"C:\Users\gayat\Downloads\ACV_projects-20260306T053026Z-1-001\ACV_GESTURE PROJECT\hand_landmarker.task"
PINCH_THRESHOLD = 0.04
AIR_CONF_THRESHOLD = 0.75
RESAMPLE_POINTS = 100

# ================= HAND TRACKER =================
base_options = python.BaseOptions(model_asset_path=HAND_MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    running_mode=vision.RunningMode.IMAGE
)

detector = vision.HandLandmarker.create_from_options(options)

# ================= AIR PREPROCESS =================
def preprocess_stroke(points, num_points=100):
    if len(points) < 20:
        return None

    pts = np.array(points, dtype=np.float32)
    pts -= np.mean(pts, axis=0)
    pts /= (np.max(np.abs(pts)) + 1e-6)

    t_old = np.linspace(0, 1, len(pts))
    t_new = np.linspace(0, 1, num_points)

    x_interp = np.interp(t_new, t_old, pts[:, 0])
    y_interp = np.interp(t_new, t_old, pts[:, 1])

    return np.stack((x_interp, y_interp), axis=1).flatten()

# ================= MAIN =================
cap = cv2.VideoCapture(0)

points = []
output_text = ""
mode = "AIR"  # AIR or GESTURE

print("Controls:")
print("F - Switch Mode")
print("E - Confirm Letter (Air Mode)")
print("C - Clear Text")
print("Q - Quit")

while True:
    start_time = time.time()
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    predicted_gesture = None
    pinch_active = False

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:

            h, w, _ = frame.shape

            # ================= PINCH =================
            thumb = hand_landmarks[4]
            index = hand_landmarks[8]

            dist = math.sqrt(
                (thumb.x - index.x)**2 +
                (thumb.y - index.y)**2
            )

            if dist < PINCH_THRESHOLD:
                pinch_active = True

            # ================= LANDMARKS =================
            landmarks = []
            for lm in hand_landmarks:
                landmarks.append([lm.x, lm.y])

            landmarks = np.array(landmarks, dtype=np.float32)

            # Flip left hand
            if result.handedness:
                hand_label = result.handedness[0][0].category_name
                if hand_label == "Left":
                    landmarks[:, 0] = 1.0 - landmarks[:, 0]

            # Gesture prediction
            features = landmarks.flatten().reshape(1, -1)
            features = gesture_scaler.transform(features)

            scores = gesture_model.decision_function(features)
            best_index = np.argmax(scores)
            predicted_gesture = GESTURE_CLASSES[best_index]

            # Draw landmarks
            for lm in hand_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)

    # ================= AIR MODE =================
    if mode == "AIR":
        if pinch_active and result.hand_landmarks:
            hand_landmarks = result.hand_landmarks[0]
            h, w, _ = frame.shape
            index = hand_landmarks[8]
            ix, iy = int(index.x * w), int(index.y * h)
            points.append((ix, iy))

        for i in range(1, len(points)):
            cv2.line(frame, points[i-1], points[i], (255, 0, 0), 4)

        mode_text = "MODE: AIR WRITING"

    # ================= GESTURE MODE =================
    else:
        mode_text = "MODE: GESTURE RECOGNITION"

        if predicted_gesture:
            cv2.putText(frame,
                        f"Gesture: {predicted_gesture}",
                        (20, 120),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9,
                        (0, 255, 0),
                        2)

    # ================= UI =================
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (40, 40, 40), -1)

    cv2.putText(frame, mode_text,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2)

    latency = (time.time() - start_time) * 1000
    cv2.putText(frame, f"Latency: {latency:.1f} ms",
                (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 200, 200),
                2)

    # Output at bottom
    cv2.putText(frame, output_text,
                (50, frame.shape[0] - 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (0, 0, 255),
                3)

    cv2.imshow("Hybrid Air Writing + Gesture System", frame)

    key = cv2.waitKey(1) & 0xFF

    # ================= KEY CONTROLS =================
    if key == ord('f'):
        mode = "GESTURE" if mode == "AIR" else "AIR"
        points = []

    elif key == ord('e') and mode == "AIR":
        features = preprocess_stroke(points, RESAMPLE_POINTS)
        if features is not None:
            features = air_scaler.transform([features])
            probs = air_model.predict_proba(features)[0]
            if np.max(probs) > AIR_CONF_THRESHOLD:
                letter = air_model.classes_[np.argmax(probs)]
                output_text += letter
        points = []

    elif key == ord('c'):
        output_text = ""
        points = []

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Controls:
F - Switch Mode
E - Confirm Letter (Air Mode)
C - Clear Text
Q - Quit
